# Systolic Array 矩阵乘法加速器 — 板级基准测试

在领航者 ZYNQ-7020 上测试 Systolic Array 加速器, 测量时延并图形化显示。

**产出 4 张图:**
1. 时延 vs 矩阵尺寸 (柱状图)
2. 吞吐量 vs 矩阵尺寸 (折线图)
3. 64×64 结果矩阵热力图
4. HW vs SW 误差分布直方图

## 1. 加载 Overlay + 驱动

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

# 驱动文件在上级目录 (按实际路径调整)
sys.path.insert(0, '/home/xilinx/pynq/overlays/matmul_systolic')
from pynq_driver import MatMulAccelerator

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11

accel = MatMulAccelerator('/home/xilinx/pynq/overlays/matmul_systolic/design_1.bit')
print('驱动就绪')

## 2. 功能自检 (16×16)

In [ ]:
np.random.seed(42)
A = (np.random.rand(16, 16).astype(np.float32) - 0.5) * 4
B = (np.random.rand(16, 16).astype(np.float32) - 0.5) * 4
C_hw, lat = accel.matmul(A, B)
C_sw = np.dot(A, B)
err = np.abs(C_hw - C_sw)
print(f'16×16: 延迟 {lat:.2f} ms, 最大误差 {err.max():.4f}, '
      f'{"PASS" if err.max() < 0.03 else "FAIL"}')

## 3. 基准测试 (多尺寸 × 多次运行)

In [ ]:
sizes = [16, 32, 48, 64]
results = accel.benchmark(sizes=sizes, runs=5)
print('\n基准测试完成')

## 4. 图表 1 — 时延 vs 矩阵尺寸 (柱状图)

In [ ]:
fig, ax = plt.subplots()
lat = [results[s]['latency_ms'] for s in sizes]
lat_std = [results[s]['latency_std'] for s in sizes]
bars = ax.bar([str(s) for s in sizes], lat, yerr=lat_std, capsize=5,
              color='#4C72B0', edgecolor='black', alpha=0.85)
ax.set_xlabel('Matrix Size (M=N=K)')
ax.set_ylabel('Latency (ms)')
ax.set_title('Hardware Latency vs Matrix Size (Systolic Array @100MHz)')
ax.grid(axis='y', alpha=0.3)
for bar, v in zip(bars, lat):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.3,
            f'{v:.2f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('latency_bars.png', dpi=120)
plt.show()

## 5. 图表 2 — 吞吐量 vs 矩阵尺寸 (折线图)

In [ ]:
fig, ax = plt.subplots()
gops = [results[s]['gops'] for s in sizes]
ax.plot(sizes, gops, 'o-', color='#DD8452', linewidth=2, markersize=8)
ax.set_xlabel('Matrix Size (M=N=K)')
ax.set_ylabel('Throughput (GOPS)')
ax.set_title('Hardware Throughput vs Matrix Size')
ax.grid(alpha=0.3)
for s, g in zip(sizes, gops):
    ax.annotate(f'{g:.2f}', (s, g), textcoords='offset points',
                xytext=(0, 10), ha='center', fontsize=10)
# 理论峰值: 16 MAC/cycle × 100MHz = 1.6 GOPS
ax.axhline(y=1.6, color='r', linestyle='--', alpha=0.5, label='Peak 1.6 GOPS (16MAC×100MHz)')
ax.legend()
plt.tight_layout()
plt.savefig('throughput_curve.png', dpi=120)
plt.show()

## 6. 图表 3 — 64×64 结果矩阵热力图

In [ ]:
np.random.seed(42)
A64 = (np.random.rand(64, 64).astype(np.float32) - 0.5) * 2
B64 = (np.random.rand(64, 64).astype(np.float32) - 0.5) * 2
C64_hw, lat64 = accel.matmul(A64, B64)
C64_sw = np.dot(A64, B64)
print(f'64×64: 延迟 {lat64:.2f} ms')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
vmax = max(abs(C64_hw).max(), abs(C64_sw).max())
im0 = axes[0].imshow(C64_hw, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0].set_title('Hardware (Q8.8 Systolic)')
im1 = axes[1].imshow(C64_sw, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[1].set_title('Software (numpy float32)')
err64 = C64_hw - C64_sw
im2 = axes[2].imshow(err64, cmap='RdBu_r', vmin=-0.05, vmax=0.05)
axes[2].set_title(f'Error (max={abs(err64).max():.4f})')
for im, ax in zip([im0, im1, im2], axes):
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig('result_heatmap.png', dpi=120)
plt.show()

## 7. 图表 4 — HW vs SW 误差分布直方图

In [ ]:
fig, ax = plt.subplots()
errs = (err64.flatten() * 256)   # 转换为 LSB (1 LSB = 1/256)
ax.hist(errs, bins=50, color='#55A868', edgecolor='black', alpha=0.85)
ax.axvline(x=0, color='r', linestyle='--', alpha=0.7)
ax.set_xlabel('Error (LSB, 1 LSB = 1/256 ≈ 0.004)')
ax.set_ylabel('Count')
ax.set_title(f'HW vs SW Error Distribution (64×64, {len(errs)} elements)\n'
             f'mean={errs.mean():.2f} LSB, std={errs.std():.2f} LSB, '
             f'max={abs(errs).max():.1f} LSB')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('error_histogram.png', dpi=120)
plt.show()

## 8. 结果汇总表

In [ ]:
print(f"{'Size':>6} | {'Latency(ms)':>11} | {'Std(ms)':>8} | {'GOPS':>7} | {'MaxErr':>8} | {'Status':>6}")
print('-' * 62)
for s in sizes:
    r = results[s]
    st = 'PASS' if r['pass'] else 'FAIL'
    print(f"{s:>6} | {r['latency_ms']:>11.2f} | {r['latency_std']:>8.2f} | "
          f"{r['gops']:>7.2f} | {r['max_err']:>8.4f} | {st:>6}")

## 9. (可选) 软件基准对比

In [ ]:
import time
sw_lat = {}
for s in sizes:
    A = (np.random.rand(s, s).astype(np.float32) - 0.5) * 4
    B = (np.random.rand(s, s).astype(np.float32) - 0.5) * 4
    t0 = time.perf_counter()
    for _ in range(5):
        np.dot(A, B)
    sw_lat[s] = (time.perf_counter() - t0) / 5 * 1000

fig, ax = plt.subplots()
x = np.arange(len(sizes))
w = 0.35
hw = [results[s]['latency_ms'] for s in sizes]
sw = [sw_lat[s] for s in sizes]
ax.bar(x - w/2, hw, w, label='Hardware (Systolic)', color='#4C72B0')
ax.bar(x + w/2, sw, w, label='Software (numpy)', color='#DD8452')
ax.set_xticks(x)
ax.set_xticklabels([str(s) for s in sizes])
ax.set_xlabel('Matrix Size')
ax.set_ylabel('Latency (ms)')
ax.set_title('Hardware vs Software Latency')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('hw_vs_sw.png', dpi=120)
plt.show()